# Pittsburgh street-closure category analysis
OpenRouter experiment scaffold matching the proposal: 6 categories, balanced sample, prompt-development split, held-out test split, multiple models, and 0/1/2-shot comparisons.


In [ ]:
import os, pandas as pd, matplotlib.pyplot as plt
from road_ai import *
DATA_PATH = "pgh data.csv"
LABEL_COL = "work_type"
CATEGORIES = DEFAULT_CATEGORIES
MODELS = ["meta-llama/llama-4-scout", "openai/gpt-oss-120b"]
df = pd.read_csv(DATA_PATH)
df.head()


## Data audit
The WPRDC dataset can repeat one permit across multiple roadway segments. De-duplicate before sampling/splitting so the same description does not leak into prompt development and final testing.


In [ ]:
print(df.shape)
print(df.columns.tolist())
print("Unique permits:", df.permit_id.nunique())
print("Unique descriptions:", df.work_description.nunique())
print(df[["from_date","to_date"]].isna().mean())


## Build balanced 300-example corpus
This will only run once `LABEL_COL` points to the field containing the six proposal categories in the full dataset.


In [ ]:
sample = balanced_sample(df, LABEL_COL, CATEGORIES, n_per_category=50, random_state=42)
prompt_dev, test = split_prompt_test(sample, LABEL_COL, prompt_n=120, test_n=180, random_state=42)
print(prompt_dev[LABEL_COL].value_counts())
print(test[LABEL_COL].value_counts())


## Compare models and prompt conditions
Every model sees the exact same held-out 180 rows. `shots=0` lists only allowed labels; 1 and 2 add labeled examples per class sampled from prompt-development data.


In [ ]:
all_results=[]
for model in MODELS:
    for shots in [0,1,2]:
        examples=make_examples(prompt_dev, LABEL_COL, k_per_class=shots)
        r=run_experiment(test, LABEL_COL, model=model, categories=CATEGORIES, examples=examples)
        r["model"]=model; r["shots"]=shots
        all_results.append(r)
results=pd.concat(all_results,ignore_index=True)
results.to_csv("classification_results.csv",index=False)
results.head()


In [ ]:
summary=[]
for (model,shots),g in results.groupby(["model","shots"]):
    s=score_results(g,CATEGORIES)
    summary.append({"model":model,"shots":shots,"accuracy":s["accuracy"],"macro_f1":s["macro_f1"],"errors":s["n_errors"],
                    "prompt_tokens":g.prompt_tokens.sum(),"completion_tokens":g.completion_tokens.sum()})
summary=pd.DataFrame(summary).sort_values(["model","shots"])
display(summary)


In [ ]:
# Confusion matrix for any selected run
MODEL=MODELS[0]; SHOTS=0
g=results[(results.model==MODEL)&(results.shots==SHOTS)]
s=score_results(g,CATEGORIES)
display(s["confusion_matrix"])
ax=s["confusion_matrix"].plot(kind="bar",figsize=(11,5),title=f"Confusions: {MODEL}, {SHOTS}-shot")
plt.ylabel("Count"); plt.tight_layout()


## Error review
Review low-confidence errors and disagreements across models. This is where you can distinguish genuine model failure from descriptions that do not contain enough evidence to justify the recorded label.


In [ ]:
errors=results[results.actual != results.predicted].sort_values(["model","shots","confidence"])
display(errors[["model","shots","permit_id","description","actual","predicted","confidence","reason"]].head(50))
